# Benzene CAS(4,4): counterdiabatic VQE and noise benchmarks

Companion notebook for **Resource-Efficient Counterdiabatic VQE in Symmetry-Reduced Active Spaces**.  
Preprint: [arXiv link — to be updated](https://arxiv.org/abs/XXXX.XXXXX)

This workflow uses symmetry-reduced qubit Hamiltonians for benzene, constructs the CD-full and CD-3 circuits, optimizes several reference ansätze, and evaluates their depolarizing-noise robustness.

**Reproducibility notes**
- The notebook reads precomputed Hamiltonians from `hami-data/`; **these files must be supplied** for the full scan.
- Molecular distances are the C–H distances in ångströms; the C–C distance is scaled with the same factor.
- CD-full is constructed from the first commutator and retains Pauli words of weight at most five (the cutoff in the research notebook); CD-3 retains weight at most three.
- The CD ansätze preserve the original `X`-on-qubit-0 reference and the 200-step MindSpore Adam optimization. Adaptive-CD, SAE-UCCSD, and HEA use the companion module.
- Noisy measurements are performed **after** optimizing the ideal circuits; no noise-aware reoptimization is performed.
- This notebook reports simulator-level gate counts, **not** ISA-transpiled hardware resources.

## 1. Imports and experiment configuration

In [ ]:
from pathlib import Path
from collections import OrderedDict
from math import cos, sin, pi

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mindspore.nn as nn
from mindquantum import Simulator
from mindquantum.core.circuit import Circuit, decompose_single_term_time_evolution
from mindquantum.core.gates import X
from mindquantum.core.operators import Hamiltonian, QubitOperator, commutator
from mindquantum.framework import MQAnsatzOnlyLayer
from qiskit.quantum_info import SparsePauliOp
from pyscf import mcscf, scf, gto
import quantumsymmetry as qs

import ansatz_noise_benchmark_adam as anb

BASIS = "sto-3g"
CAS = (4, 4)
SPIN = 0
CHARGE = 0
BOND_DISTANCES = np.linspace(0.4, 3.0, 25)
HAMILTONIAN_DIR = Path("hami-data")
OPTIMIZER_STEPS = 200
LEARNING_RATE = 0.01
OUTPUT_DIR = Path("results")
OUTPUT_DIR.mkdir(exist_ok=True)

## 2. Benzene geometry and symmetry-reduced Hamiltonians

We retain the original geometry convention used to produce the archived Hamiltonian files. The C–C bond length is `(1.39 / 1.09) × R`, where `R` is the C–H bond length.

In [ ]:
def geometry(z):
    a  = 1.39
    b  = 1.09
    cc = a/b*z
    # Original molecule
    atom = [
        ['C', (-cos(np.pi/3)*cc, sin(np.pi/3)*cc, 0.00)],
        ['C', ( cos(np.pi/3)*cc, sin(np.pi/3)*cc, 0.00)],
        ['C', (             -cc,            0.00, 0.00)],
        ['C', (              cc,            0.00, 0.00)],
        ['C', (-cos(np.pi/3)*cc,-sin(np.pi/3)*cc, 0.00)],
        ['C', ( cos(np.pi/3)*cc,-sin(np.pi/3)*cc, 0.00)],
        ['H', ( cos(np.pi/3)*(cc+z),  sin(np.pi/3)*(cc+z), 0.00)],
        ['H', (-cos(np.pi/3)*(cc+z),  sin(np.pi/3)*(cc+z), 0.00)],
        ['H', (-cos(np.pi/3)*(cc+z), -sin(np.pi/3)*(cc+z), 0.00)],
        ['H', ( cos(np.pi/3)*(cc+z), -sin(np.pi/3)*(cc+z), 0.00)],
        ['H', (             -(cc+z),                 0.00, 0.00)],
        ['H', (              (cc+z),                 0.00, 0.00)]
    ]
    return atom

In [ ]:
qubit_hamiltonian_record = []
encoded_exact_energies = []
hf_circuits = []
atom_record = []

# Fail explicitly rather than silently skipping unavailable geometries:
# a skipped geometry would misalign Hamiltonians and bond-distance labels.
missing = [
    HAMILTONIAN_DIR / f"Benzene_(4,4){bond:.3f}.dat"
    for bond in BOND_DISTANCES
    if not (HAMILTONIAN_DIR / f"Benzene_(4,4){bond:.3f}.dat").is_file()
]
if missing:
    raise FileNotFoundError(
        f"Missing {len(missing)} Hamiltonian files in {HAMILTONIAN_DIR}/. "
        f"For example: {missing[0]}. Supply the precomputed data before running."
    )

for bond in BOND_DISTANCES:
    path = HAMILTONIAN_DIR / f"Benzene_(4,4){bond:.3f}.dat"
    data = np.loadtxt(path, dtype=str, ndmin=2)
    qham = SparsePauliOp(data[:, 0].tolist(), data[:, 1].astype(float))
    qubit_hamiltonian_record.append(qham)

    e0, _ = anb.get_exact_ground_state(qham)
    encoded_exact_energies.append(e0)

    atom = geometry(bond)
    atom_record.append(atom)
    enc = qs.make_encoding(atom=atom, basis=BASIS, CAS=CAS)
    hf_circuits.append(qs.HartreeFockCircuit(enc, atom=atom, basis=BASIS, CAS=CAS))
    print(f"R = {bond:.3f} Å | {qham.num_qubits} qubits | E0 = {e0:.10f} Ha")

## 3. CD-full and CD-3 ansätze

The ansatz starts from `X.on(0)`, following the reference notebook. The candidate pool comes from `[Z_0 - Z_1 - …, H]`; each retained Pauli word is assigned an independent variational parameter. The CD-full label denotes the *weight-five cutoff used here*, not an unrestricted all-orders adiabatic gauge potential.

In [ ]:
def make_cd_circuit(cd_pool):
    """Append one variational Pauli-evolution gate per selected CD word.

    Keep the reference preparation X(0) for consistency with the paper's
    reported CD optimization. This differs from the explicit encoded-HF
    preparation used for other ansatz families.
    """
    circuit = Circuit()
    circuit += X.on(0)
    for idx, term in enumerate(cd_pool):
        circuit += decompose_single_term_time_evolution(term, {f"p{idx}": 1.0})
    return circuit


def optimize_cd_with_mindspore(qham, circuit):
    """Reference training protocol: MindSpore Adam, 200 gradient steps.

    The network's default parameter initialization is preserved. Setting a
    deterministic seed or switching to zero initialization changes the
    optimizer trajectory and must be treated as a new numerical experiment.
    """
    ham_mq = Hamiltonian(anb.generate_hamiltonian(qham))
    simulator = Simulator("mqvector", qham.num_qubits)
    gradient_operator = simulator.get_expectation_with_grad(ham_mq, circuit)
    network = MQAnsatzOnlyLayer(gradient_operator)
    optimizer = nn.Adam(network.trainable_params(), learning_rate=LEARNING_RATE)
    training_step = nn.TrainOneStepCell(network, optimizer)
    energy_history = []
    for _ in range(OPTIMIZER_STEPS):
        energy = training_step()
        energy_history.append(float(np.asarray(energy.asnumpy()).reshape(-1)[0]))
    params = dict(zip(circuit.params_name, map(float, network.weight.asnumpy())))
    return params, energy_history


def run_cd_scan(max_pauli_weight):
    """Optimize a fixed CD ansatz at each molecular geometry."""
    circuits, params, energies, pools = [], [], [], []
    for idx, qham in enumerate(qubit_hamiltonian_record):
        # The default reference Z coefficients are (+1, -1, -1, ...).
        pool = anb.build_cd_pool(qham, interaction_order=max_pauli_weight)
        circuit = make_cd_circuit(pool)
        parameter_values, energy_history = optimize_cd_with_mindspore(qham, circuit)
        pools.append(pool)
        circuits.append(circuit)
        params.append(parameter_values)
        energies.append(energy_history[-1])
        print(
            f"[CD-{max_pauli_weight}] {idx + 1}/{len(qubit_hamiltonian_record)} "
            f"operators={len(pool)}, E={energy_history[-1]:.10f} Ha"
        )
    return circuits, params, energies, pools


cd_circuit_record_full, cd_pr_full, cd_energies_full, cd_record_full = run_cd_scan(5)
circuit_cd_record_3, pr_record_3, cd_energies_3, cd_record_3 = run_cd_scan(3)

## 4. Adaptive-CD-3 and Adaptive-CD-4

The gradient criterion and reoptimization are implemented in `ansatz_noise_benchmark_adam.py`. The pool is generated from the same reference commutator, then filtered by Pauli weight. Operators can be selected repeatedly. The original maximum of 50 adaptive steps and Adam settings are preserved.

In [ ]:
hf_circuit_record = [anb.qiskit_to_mq_circuit(hf) for hf in hf_circuits]

adaptive_circuits_3, adaptive_params_3, adaptive_energies_3, adaptive_metadata_3 = (
    anb.batch_adaptive_cd(
        hf_circuit_record=hf_circuit_record,
        qubit_hamiltonian_record=qubit_hamiltonian_record,
        cd_pool_record=cd_record_full,
        interaction_order=3,
        gradient_threshold=1e-3,
        max_adapt_iterations=50,
        optimizer="Adam",
        optimizer_maxiter=OPTIMIZER_STEPS,
        learning_rate=LEARNING_RATE,
        optimizer_tol=None,
        allow_repeated_operators=True,
        checkpoint_file=str(OUTPUT_DIR / "adaptive_cd_3_ideal.csv"),
        verbose=True,
    )
)

adaptive_circuits_4, adaptive_params_4, adaptive_energies_4, adaptive_metadata_4 = (
    anb.batch_adaptive_cd(
        hf_circuit_record=hf_circuit_record,
        qubit_hamiltonian_record=qubit_hamiltonian_record,
        cd_pool_record=cd_record_full,
        interaction_order=4,
        gradient_threshold=1e-3,
        max_adapt_iterations=50,
        optimizer="Adam",
        optimizer_maxiter=OPTIMIZER_STEPS,
        learning_rate=LEARNING_RATE,
        optimizer_tol=None,
        allow_repeated_operators=True,
        checkpoint_file=str(OUTPUT_DIR / "adaptive_cd_4_ideal.csv"),
        verbose=True,
    )
)

## 5. Reference ansätze: SAE-UCCSD and HEA

The SAE-UCCSD excitation pool is mapped using the symmetry-aware encoding; the HEA depth is selected to approximately match the parameter count of CD-full. Both use 200 Adam steps in the module implementation.

In [ ]:
uccsd_circuits, uccsd_params, uccsd_energies, uccsd_metadata = (
    anb.batch_sae_uccsd(
        atom_record=atom_record,
        basis=BASIS,
        CAS=CAS,
        qubit_hamiltonian_record=qubit_hamiltonian_record,
        charge=CHARGE,
        spin=SPIN,
        encoding_type="SAE",
        method="Adam",
        maxiter=OPTIMIZER_STEPS,
        learning_rate=LEARNING_RATE,
        tol=None,
        checkpoint_file=str(OUTPUT_DIR / "uccsd_ideal.csv"),
        verbose=True,
    )
)

hf_mq_record = anb.make_sae_hf_record(atom_record=atom_record, basis=BASIS, CAS=CAS)
hea_circuit_record, hea_pr_record, hea_energy_record, hea_depth_record = anb.batch_hea(
    hf_circuit_record=hf_mq_record,
    qubit_hamiltonian_record=qubit_hamiltonian_record,
    target_circuit_record=cd_circuit_record_full,
    method="Adam",
    maxiter=OPTIMIZER_STEPS,
    learning_rate=LEARNING_RATE,
    tol=None,
    checkpoint_file=str(OUTPUT_DIR / "hea_ideal.csv"),
    verbose=True,
)

## 6. Noise robustness benchmark

**All six ansätze remain in the benchmark** (the original notebook accidentally replaced the complete dictionary with a CD-only subset). For each optimized ideal circuit, we insert depolarizing channels with `p1 = 0.1 × p2` and compare:

- Absolute energy error `|E_noisy − E_exact|`, relative to the encoded Hamiltonian ground state.
- Noise-induced deviation `|E_noisy − E_ideal|`.
- Ground-state infidelity `1 − <ψ_exact|ρ_noisy|ψ_exact>`.

The noise scan creates `results/C6H6_ansatz_noise_benchmark.csv` and `results/C6H6_ansatz_circuit_metrics.csv`. These calculations can be computationally expensive.

In [ ]:
ansatz_records = anb.make_ansatz_records(
    cd_full_circuits=cd_circuit_record_full,
    cd_full_params=cd_pr_full,
    cd_truncated_circuits=circuit_cd_record_3,
    cd_truncated_params=pr_record_3,
    uccsd_circuits=uccsd_circuits,
    uccsd_params=uccsd_params,
    hea_circuits=hea_circuit_record,
    hea_params=hea_pr_record,
    adaptive_cd_circuits=adaptive_circuits_3,
    adaptive_cd_params=adaptive_params_3,
    truncated_label="CD-3",
    adaptive_cd_label="Adaptive-CD-3",
)
# Add the second adaptive benchmark without overwriting the first.
ansatz_records["Adaptive-CD-4"] = {
    "circuits": adaptive_circuits_4,
    "params": adaptive_params_4,
}

print("Ansätze included:", ", ".join(ansatz_records))
df_noise, df_circuit = anb.benchmark_ansatz_noise(
    qubit_hamiltonian_record=qubit_hamiltonian_record,
    ansatz_records=ansatz_records,
    p2_list=[1e-4, 2e-4, 5e-4, 1e-3, 2e-3, 5e-3, 1e-2],
    x_values=BOND_DISTANCES,
    p1_ratio=0.1,
    output_file=str(OUTPUT_DIR / "C6H6_ansatz_noise_benchmark.csv"),
    circuit_file=str(OUTPUT_DIR / "C6H6_ansatz_circuit_metrics.csv"),
    verbose=True,
)

display(df_circuit.head())
display(df_noise.head())

## 7. Optional: classical CASCI reference energies

CASCI energies refer to the chosen molecular active-space calculation, whereas the `E_exact` fields in the noise CSV are exact eigenvalues of the **encoded** Hamiltonians. They should not be conflated with full-configuration-interaction energies in the complete orbital space.

In [ ]:
casci_energies = []
for bond in BOND_DISTANCES:
    mol = gto.M(atom=geometry(bond), basis=BASIS, spin=SPIN, charge=CHARGE)
    mf = scf.HF(mol).run(verbose=0)
    solver = mcscf.CASCI(mf, CAS[1], CAS[0])
    solver.run(verbose=0)
    casci_energies.append(float(solver.e_tot))

pd.DataFrame({
    "bond_distance_A": BOND_DISTANCES,
    "encoded_exact_energy_Ha": encoded_exact_energies,
    "casci_energy_Ha": casci_energies,
}).to_csv(OUTPUT_DIR / "C6H6_reference_energies.csv", index=False)

## Interpretation and reproducibility

To reproduce published numerical values, use the original molecular Hamiltonians and the dependency versions recorded by the authors. The notebook's original MindSpore parameter initialization was not explicitly seeded, so optimization trajectories may vary across runs. The code has been reorganized for readability; numerical parity with the original research environment has **not** been independently established.

Do not confuse simulator-level circuit counts with hardware-transpiled metrics. Review the article for the exact methodology and any hardware experiments, which are not reproduced by this notebook.